## Pydantic Settings - Follow-Along Practice

This notebook covers reading and validating application configuration from environment variables using Pydantic Settings.

#### Install Dependencies

Run the following cell to install Pydantic Settings in your notebook environment.

In [ ]:
!pip install pydantic-settings

## 1. The Problem with Environment Variables

Environment variables are always read as strings in Python. This makes type coercion and conditional logic error-prone.

In [ ]:
import os

# Simulate setting environment variables
os.environ["PORT"] = "8000"
os.environ["DEBUG"] = "False"

port = os.getenv("PORT")
print("Port Type:", type(port))  # <class 'str'> - not an integer!

debug = os.getenv("DEBUG")
print("Debug Type:", type(debug))  # <class 'str'>
if debug:
    # This runs! Because "False" is a non-empty string, which evaluates to True.
    print("Debug mode active (incorrectly evaluated!)")

## 2. Basic Usage

Define a configuration class inheriting from `BaseSettings`. Pydantic automatically reads and validates corresponding environment variables.

In [ ]:
from pydantic_settings import BaseSettings
from pydantic import ValidationError

class Settings(BaseSettings):
    api_key: str
    port: int = 8000
    debug: bool = False

# Set required environment variable
os.environ["API_KEY"] = "sk_test_123"

settings = Settings()
print("API Key:", settings.api_key)
print("Port:", settings.port, type(settings.port))
print("Debug:", settings.debug, type(settings.debug))

## 3. The .env File

In development, settings are often stored in a `.env` file. You can configure `BaseSettings` to read from it using `SettingsConfigDict`.

In [ ]:
# Let's create a mock .env file first
with open(".env", "w") as f:
    f.write("API_KEY=sk_test_123\nPORT=5000\nDEBUG=True")

from pydantic_settings import SettingsConfigDict

class EnvSettings(BaseSettings):
    model_config = SettingsConfigDict(env_file=".env")
    
    api_key: str
    port: int = 8000
    debug: bool = False

# Clear env variables to ensure it reads from .env
os.environ.pop("API_KEY", None)
os.environ.pop("PORT", None)
os.environ.pop("DEBUG", None)

settings = EnvSettings()
print("API Key:", settings.api_key)  # sk_test_123
print("Port:", settings.port)     # 5000 (parsed as integer)
print("Debug:", settings.debug)    # True (parsed as boolean)

## 4. Environment Prefix

Use `env_prefix` to group and isolate application settings to avoid variable namespace collisions.

In [ ]:
# Simulate prefixed env variables
os.environ["MYAPP_API_KEY"] = "sk_prefix_abc"
os.environ["MYAPP_PORT"] = "6000"

class PrefixedSettings(BaseSettings):
    model_config = SettingsConfigDict(
        env_prefix="myapp_"
    )
    
    api_key: str
    port: int = 8000

settings = PrefixedSettings()
print("API Key:", settings.api_key)  # sk_prefix_abc
print("Port:", settings.port)        # 6000

## 5. Hiding Secrets

Wrap sensitive configuration (passwords, tokens) in `SecretStr` to prevent them from being printed or logged in plain text.

In [ ]:
from pydantic import SecretStr

class SecretSettings(BaseSettings):
    database_password: SecretStr

settings = SecretSettings(database_password="super-secret-password")

# Secrets are masked when printed
print("Printed Settings:", settings.database_password)

# Retrieve the actual value using get_secret_value()
print("Actual Value:", settings.database_password.get_secret_value())

## 6. Caching Settings

Use `lru_cache` to initialize the settings object once and reuse it across the application for better performance.

In [ ]:
from functools import lru_cache

class CacheSettings(BaseSettings):
    api_key: str

os.environ["API_KEY"] = "cached_key_123"

@lru_cache
def get_settings():
    return CacheSettings()

settings = get_settings()
print("API Key:", settings.api_key)

## 7. Environment-Specific Settings

Add custom properties to conditionalize application behavior based on the environment value.

In [ ]:
class EnvSpecificSettings(BaseSettings):
    environment: str = "development"
    debug: bool = False
    
    @property
    def is_production(self) -> bool:
        return self.environment == "production"

settings = EnvSpecificSettings(environment="production")
print("Is Production?:", settings.is_production)